# Closing the PERSON recall gap with spaCy NER - UNTESTED TEMPLATE
The rule-based detector has PERSON recall ~0.65 on held-out templates. This notebook adds spaCy NER on the same synthetic benchmark and reports the change. Needs Internet On.

In [ ]:
!pip -q install faker spacy && python -m spacy download en_core_web_sm

In [ ]:
%%writefile pii.py
"""Validator-backed PII detector + synthetic benchmark with gold spans. Original implementation."""
import re, random, json
from faker import Faker

# ---------------- validators ----------------
def luhn(s):
    d = [int(c) for c in re.sub(r"\D", "", s)][::-1]
    return len(d) >= 13 and sum(d[0::2]) + sum(sum(divmod(2 * x, 10)) for x in d[1::2]) % 10 == 0 if False else \
        (len(d) >= 13 and (sum(d[0::2]) + sum(sum(divmod(2 * x, 10)) for x in d[1::2])) % 10 == 0)

_D = [[0,1,2,3,4,5,6,7,8,9],[1,2,3,4,0,6,7,8,9,5],[2,3,4,0,1,7,8,9,5,6],[3,4,0,1,2,8,9,5,6,7],[4,0,1,2,3,9,5,6,7,8],
      [5,9,8,7,6,0,4,3,2,1],[6,5,9,8,7,1,0,4,3,2],[7,6,5,9,8,2,1,0,4,3],[8,7,6,5,9,3,2,1,0,4],[9,8,7,6,5,4,3,2,1,0]]
_P = [[0,1,2,3,4,5,6,7,8,9],[1,5,7,6,2,8,3,0,9,4],[5,8,0,3,7,9,6,1,4,2],[8,9,1,6,0,4,3,5,2,7],
      [9,4,5,3,1,2,6,8,7,0],[4,2,8,6,5,7,3,9,0,1],[2,7,9,3,8,0,6,4,1,5],[7,0,4,6,9,1,3,2,5,8]]
def verhoeff_ok(s):
    c = 0
    for i, ch in enumerate(reversed(re.sub(r"\D", "", s))): c = _D[c][_P[i % 8][int(ch)]]
    return c == 0
def verhoeff_check_digit(body):
    c = 0
    for i, ch in enumerate(reversed(body)): c = _D[c][_P[(i + 1) % 8][int(ch)]]
    return [0,4,3,2,1,5,6,7,8,9][c]

def ssn_ok(s):
    a, g, n = re.sub(r"\D", "", s)[:3], re.sub(r"\D", "", s)[3:5], re.sub(r"\D", "", s)[5:]
    return a not in ("000", "666") and not a.startswith("9") and g != "00" and n != "0000"
def ip_ok(s): return all(0 <= int(x) <= 255 for x in s.split("."))

# ---------------- detector ----------------
PAT = {
 "EMAIL": r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b",
 "CREDIT_CARD": r"\b(?:\d[ -]?){12,18}\d\b",
 "US_SSN": r"\b\d{3}-\d{2}-\d{4}\b",
 "IN_AADHAAR": r"\b[2-9]\d{3}[ -]?\d{4}[ -]?\d{4}\b",
 "IN_PAN": r"\b[A-Z]{3}[PCHFATBLJG][A-Z]\d{4}[A-Z]\b",
 "IP_ADDRESS": r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
 "PHONE": r"(?<!\w)(?:\+?\d{1,3}[ -]?)?(?:\(\d{3}\)\s?|\d{3}[ -.]?)\d{3}[ -.]?\d{4}\b|(?<!\w)(?:\+91[ -]?)?[6-9]\d{4}[ -]?\d{5}\b",
 "DATE_OF_BIRTH": r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}-\d{2}-\d{2}|\d{1,2} [A-Z][a-z]+ \d{4})\b",
}
NAME = r"([A-Z][a-z]+(?:[ ][A-Z][a-z]+){0,2})"
NAME_CTX = [r"(?:[Mm]y name is|I am|I'm|[Tt]his is|[Nn]ame:|Dear|Hi|Hello|Regards,|Thanks,|Sincerely,|Mr\.|Mrs\.|Ms\.|Dr\.|[Cc]ustomer|[Cc]ontact|[Aa]ccount holder|signed)[ ]+" + NAME]
DOB_CTX = re.compile(r"(?i)(born|dob|d\.o\.b|date of birth|birthday|birth date)")
PHONE_CTX = re.compile(r"(?i)(call|phone|mobile|tel|contact|reach|whatsapp|cell)")

def detect(text, validators=True, context=True):
    out = []
    for lab, pat in PAT.items():
        for m in re.finditer(pat, text):
            s, e, v = m.start(), m.end(), m.group()
            digits = re.sub(r"\D", "", v)
            if validators:
                if lab == "CREDIT_CARD" and not luhn(v): continue
                if lab == "IN_AADHAAR" and not (len(digits) == 12 and verhoeff_ok(v)): continue
                if lab == "US_SSN" and not ssn_ok(v): continue
                if lab == "IP_ADDRESS" and not ip_ok(v): continue
            if lab == "IP_ADDRESS" and re.search(r"(?i)(version|build|release|ver\.?|\bv)\s*$", text[max(0, s - 12):s]): continue  # added after v1 error analysis
            if context:
                win = re.split(r"[.;\n]", text[max(0, s - 40):s])[-1]  # context must be in the same clause
                if lab == "DATE_OF_BIRTH" and not DOB_CTX.search(win): continue
                if lab == "PHONE" and not PHONE_CTX.search(win) and not v.startswith("+") and "(" not in v: continue
            out.append((s, e, lab, v))
    if context:
        for pat in NAME_CTX:
            for m in re.finditer(pat, text):
                out.append((m.start(1), m.end(1), "PERSON", m.group(1)))
    # resolve overlaps: prefer more specific/longer
    pri = {"CREDIT_CARD": 5, "IN_AADHAAR": 5, "US_SSN": 5, "IN_PAN": 5, "EMAIL": 5, "IP_ADDRESS": 4,
           "DATE_OF_BIRTH": 3, "PHONE": 2, "PERSON": 3}
    out.sort(key=lambda x: (-pri[x[2]], -(x[1] - x[0])))
    kept = []
    for c in out:
        if not any(c[0] < k[1] and k[0] < c[1] for k in kept): kept.append(c)
    return sorted(kept)

def redact(text, spans=None):
    spans = detect(text) if spans is None else spans
    for s, e, lab, _ in sorted(spans, reverse=True): text = text[:s] + f"<{lab}>" + text[e:]
    return text

# ---------------- synthetic benchmark with gold spans ----------------
def gen_entities(fk, rnd):
    def luhn_card():
        b = [rnd.randint(0, 9) for _ in range(15)]; b[0] = rnd.choice([4, 5, 3])
        for c in range(10):
            if luhn("".join(map(str, b + [c]))): d = "".join(map(str, b + [c])); break
        return rnd.choice([d, " ".join(d[i:i+4] for i in range(0, 16, 4)), "-".join(d[i:i+4] for i in range(0, 16, 4))])
    def aadhaar():
        b = str(rnd.randint(2, 9)) + "".join(str(rnd.randint(0, 9)) for _ in range(10))
        d = b + str(verhoeff_check_digit(b)); return rnd.choice([d, f"{d[:4]} {d[4:8]} {d[8:]}"])
    def pan(): return "".join(rnd.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k=3)) + rnd.choice("PCHFATBLJG") + rnd.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") + f"{rnd.randint(0,9999):04d}" + rnd.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    def ssn():
        while True:
            v = f"{rnd.randint(1,899):03d}-{rnd.randint(1,99):02d}-{rnd.randint(1,9999):04d}"
            if ssn_ok(v): return v
    def phone():
        k = rnd.random()
        if k < .4: return f"+91 {rnd.randint(6,9)}{rnd.randint(1000,9999)} {rnd.randint(10000,99999)}"
        if k < .7: return f"({rnd.randint(200,999)}) {rnd.randint(200,999)}-{rnd.randint(1000,9999)}"
        return f"{rnd.randint(200,999)}-{rnd.randint(200,999)}-{rnd.randint(1000,9999)}"
    return {"PERSON": lambda: fk.first_name() + " " + fk.last_name() if rnd.random() < .8 else fk.first_name(),
            "EMAIL": lambda: fk.email(), "CREDIT_CARD": luhn_card, "IN_AADHAAR": aadhaar, "IN_PAN": pan,
            "US_SSN": ssn, "PHONE": phone, "IP_ADDRESS": lambda: fk.ipv4(),
            "DATE_OF_BIRTH": lambda: rnd.choice([fk.date_of_birth().strftime("%d/%m/%Y"), fk.date_of_birth().strftime("%Y-%m-%d"), fk.date_of_birth().strftime("%d %B %Y")])}

def hard_negatives(fk, rnd):
    """Strings that LOOK like PII but are not."""
    def bad_card():
        while True:
            d = "".join(str(rnd.randint(0, 9)) for _ in range(16))
            if not luhn(d): return d
    return [("order id " + bad_card(), 0), ("tracking " + str(rnd.randint(10**9, 10**10)), 0),
            ("version " + ".".join(str(rnd.randint(0, 20)) for _ in range(4)), 0),
            ("build " + f"{rnd.randint(0,255)}.{rnd.randint(256,999)}.{rnd.randint(0,255)}.{rnd.randint(0,255)}", 0),
            ("ticket 000-12-3456", 0), ("ref 987-65-4321", 0),
            ("released on " + fk.date().replace("-", "/"), 0),
            ("The Jordan sneakers and the Apple Watch", 0)]

# Disjoint template sets: DEV used while writing rules, TEST (held out) used only for the reported numbers.
DEV_T = ["Hi, my name is {PERSON}. You can reach me at {EMAIL} or call {PHONE}.",
         "Customer {PERSON} paid with card {CREDIT_CARD}. Born {DATE_OF_BIRTH}.",
         "Aadhaar {IN_AADHAAR} and PAN {IN_PAN} were submitted by {PERSON}.",
         "Login from {IP_ADDRESS} for {EMAIL}. SSN on file {US_SSN}."]
TEST_T = ["Dear {PERSON}, we could not verify {IN_AADHAAR}. Please call us on {PHONE} before the deadline.",
          "Ticket opened by {PERSON}: user {EMAIL} reported a failed charge on {CREDIT_CARD} from {IP_ADDRESS}.",
          "Applicant details - PAN: {IN_PAN}, date of birth {DATE_OF_BIRTH}, contact {PHONE}. Regards, {PERSON}",
          "Hello team, this is {PERSON}. My SSN is {US_SSN} and I was born {DATE_OF_BIRTH}.",
          "Account holder {PERSON} ({EMAIL}) asked about invoice charges. WhatsApp {PHONE}.",
          "Signed {PERSON}. Bank card {CREDIT_CARD}, mobile {PHONE}."]

def make_doc(templates, fk, rnd, n_neg=2):
    ents = gen_entities(fk, rnd); t = rnd.choice(templates); text, gold = "", []
    i = 0
    for m in re.finditer(r"\{(\w+)\}", t):
        text += t[i:m.start()]; v = ents[m.group(1)]()
        gold.append((len(text), len(text) + len(v), m.group(1), v)); text += v; i = m.end()
    text += t[i:]
    neg = rnd.sample(hard_negatives(fk, rnd), n_neg)
    text += "\nNotes: " + "; ".join(x[0] for x in neg) + "."
    return text, gold

def score(docs, **kw):
    tp = {}; fp = {}; fn = {}
    for text, gold in docs:
        pred = detect(text, **kw); g = {(s, e, l) for s, e, l, _ in gold}; p = {(s, e, l) for s, e, l, _ in pred}
        for l in {x[2] for x in g | p}:
            gl, pl = {x for x in g if x[2] == l}, {x for x in p if x[2] == l}
            tp[l] = tp.get(l, 0) + len(gl & pl); fp[l] = fp.get(l, 0) + len(pl - gl); fn[l] = fn.get(l, 0) + len(gl - pl)
    r = {}
    for l in sorted(tp):
        P = tp[l] / max(tp[l] + fp[l], 1); R = tp[l] / max(tp[l] + fn[l], 1)
        r[l] = {"precision": round(P, 3), "recall": round(R, 3), "f1": round(2 * P * R / max(P + R, 1e-9), 3), "tp": tp[l], "fp": fp[l], "fn": fn[l]}
    T, F, N = sum(tp.values()), sum(fp.values()), sum(fn.values())
    P, R = T / max(T + F, 1), T / max(T + N, 1)
    r["MICRO"] = {"precision": round(P, 3), "recall": round(R, 3), "f1": round(2 * P * R / max(P + R, 1e-9), 3), "tp": T, "fp": F, "fn": N}
    return r

if __name__ == "__main__":
    fk = Faker(["en_US", "en_IN"]); Faker.seed(2024); rnd = random.Random(2024)  # fresh seed: v1 and v2 test sets were used for error analysis
    dev = [make_doc(DEV_T, fk, rnd) for _ in range(400)]
    test = [make_doc(TEST_T, fk, rnd) for _ in range(1500)]
    out = {"n_test_docs": len(test), "n_gold_spans": sum(len(g) for _, g in test),
           "ablation_micro": {}, "full_per_entity_heldout_templates": None, "dev_micro": score(dev)["MICRO"]}
    for name, kw in {"regex_only": dict(validators=False, context=False), "regex+validators": dict(validators=True, context=False),
                     "regex+validators+context": dict(validators=True, context=True)}.items():
        out["ablation_micro"][name] = score(test, **kw)["MICRO"]
    out["full_per_entity_heldout_templates"] = score(test)
    json.dump(out, open("results/pii_metrics.json", "w"), indent=1)
    json.dump([{"text": t, "gold": g} for t, g in test[:50]], open("results/sample_docs.json", "w"), indent=1)
    print(json.dumps(out["ablation_micro"], indent=1)); print(json.dumps(out["full_per_entity_heldout_templates"], indent=1))


In [ ]:
import spacy, random, pii
from faker import Faker
nlp = spacy.load("en_core_web_sm")
fk = Faker(["en_US", "en_IN"]); Faker.seed(31337); rnd = random.Random(31337)   # NEW seed: not used in any earlier error analysis
docs = [pii.make_doc(pii.TEST_T, fk, rnd) for _ in range(1500)]
def hybrid(text, **kw):
    spans = [s for s in pii.detect(text) if s[2] != "PERSON"]
    for e in nlp(text).ents:
        if e.label_ == "PERSON": spans.append((e.start_char, e.end_char, "PERSON", e.text))
    return sorted(spans)
orig = pii.detect; base = pii.score(docs)["PERSON"]; pii.detect = lambda t, **kw: hybrid(t)
print("rules only:", base); print("rules + spaCy NER:", pii.score(docs)["PERSON"]); pii.detect = orig